# VacationPy
---

## Starter Code to Import Libraries and Load the Weather and Coordinates Data

In [5]:
# Dependencies and Setup
import hvplot.pandas
import pandas as pd
import requests
from pprint import pprint

# Import API key
from api_keys import geoapify_key

In [6]:
# Load the CSV file created in Part 1 into a Pandas DataFrame
city_data_df = pd.read_csv("output_data/cities.csv")

# Display sample data
city_data_df.head()

,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
0,0,nar'yan-mar,67.6713,53.0870,-5.97,86,9,7.03,RU,1744183406
1,1,crane,31.3974,-102.3501,16.55,20,0,6.88,US,1744183407
2,2,manacapuru,-3.2997,-60.6206,24.01,95,100,1.07,BR,1744183408
3,3,blackmans bay,-43.0167,147.3167,16.96,64,0,2.40,AU,1744183409
4,4,puerto natales,-51.7236,-72.4875,7.48,87,100,0.79,CL,1744183410


---

### Step 1: Create a map that displays a point for every city in the `city_data_df` DataFrame. The size of the point should be the humidity in each city.

In [8]:
%%capture --no-display

import hvplot.pandas

# Configure the map plot
map_plot = city_data_df.hvplot.points(
    "Lng",                       # Longitude
    "Lat",                       # Latitude
    geo=True,                    # Enable geographic plotting
    tiles='OSM',                 # Add OpenStreetMap tiles as background
    frame_width=700,             # Width of the plot
    frame_height=500,            # Height of the plot
    size="Humidity",             # Point size based on humidity
    scale=0.5,                   # Scale factor for the point size
    color="City",                # Color by city
    hover_cols=["City", "Country", "Max Temp"],  # Columns to show in the tooltip
)

# Display the map plot
map_plot



:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity,Country,Max Temp)

### Step 2: Narrow down the `city_data_df` DataFrame to find your ideal weather condition

In [10]:
# Narrow down cities based on ideal weather conditions (e.g., temp between 20°C and 30°C, humidity below 70%)
filtered_cities_df = city_data_df[(city_data_df["Max Temp"] >= 20) & 
                                   (city_data_df["Max Temp"] <= 30) & 
                                   (city_data_df["Humidity"] < 70)]

# Drop any rows with null values
filtered_cities_df = filtered_cities_df.dropna()

# Display sample data
filtered_cities_df.head()


,City_ID,City,Lat,Lng,Max Temp,Humidity,Cloudiness,Wind Speed,Country,Date
6,6,mashava,-20.0367,30.4823,26.71,43,3,2.67,ZW,1744183414
10,10,yicheng,31.3705,119.8703,27.63,40,100,3.26,CN,1744183418
16,16,buka,40.8108,69.1986,21.64,52,100,2.47,UZ,1744183425
60,60,ma'an,26.9775,110.7211,25.88,59,100,1.34,CN,1744183475
67,67,reggane,26.7158,0.1714,21.35,23,0,7.62,DZ,1744183484


### Step 3: Create a new DataFrame called `hotel_df`.

In [11]:
# Create a new DataFrame called hotel_df by copying the necessary columns from the filtered DataFrame
hotel_df = filtered_cities_df[["City", "Country", "Lat", "Lng", "Humidity"]].copy()

# Add an empty column "Hotel Name" to store the hotel found using the Geoapify API
hotel_df["Hotel Name"] = ""

# Display sample data to verify the new DataFrame
hotel_df.head()


,City,Country,Lat,Lng,Humidity,Hotel Name
6,mashava,ZW,-20.0367,30.4823,43,
10,yicheng,CN,31.3705,119.8703,40,
16,buka,UZ,40.8108,69.1986,52,
60,ma'an,CN,26.9775,110.7211,59,
67,reggane,DZ,26.7158,0.1714,23,


### Step 4: For each city, use the Geoapify API to find the first hotel located within 10,000 metres of your coordinates.

In [18]:
import requests
from pprint import pprint

# Initialize an empty list to store hotel rows
hotel_rows = []

# Set the parameters for the type of place
categories = "accommodation.hotel"
limit = 1  # Only get the closest hotel
radius = 30000  # Increase radius to 30,000 meters (30 km) for broader search

# Iterate through the hotel_df DataFrame
for index, row in hotel_df.iterrows():
    # Get latitude, longitude, and city details from the DataFrame
    city = row["City"]
    country = row["Country"]
    latitude = row["Lat"]
    longitude = row["Lng"]
    humidity = row["Humidity"]

    # Set the parameters for the search
    filters = f"circle:{longitude},{latitude},{radius}"
    bias = f"proximity:{longitude},{latitude}"
    
    params = {
        "categories": categories,
        "limit": limit,
        "filter": filters,
        "bias": bias,
        "apiKey": geoapify_key  # Make sure to replace with your actual Geoapify API key
    }

    # Print a message to indicate the start of the hotel search for the current city
    pprint(f"Starting hotel search for {city}, {country}")

    # Set base URL for Geoapify API request
    base_url = "https://api.geoapify.com/v2/places"

    # Make the API request using the parameters dictionary
    response = requests.get(base_url, params=params)

    # Convert the API response to JSON format
    response_json = response.json()

    # Print the response JSON to troubleshoot
    pprint(response_json)  # This will show you the structure of the returned data

    # Check if any features were returned in the response
    if response_json.get("features"):
        try:
            # Extract hotel details if available
            hotel_df.loc[index, "Hotel Name"] = response_json["features"][0]["properties"]["name"]
            hotel_df.loc[index, "Hotel Address"] = response_json["features"][0]["properties"]["address_line1"]
            hotel_df.loc[index, "Hotel Distance"] = response_json["features"][0]["properties"]["distance"]
            hotel_df.loc[index, "Hotel Website"] = response_json["features"][0]["properties"]["datasource"]["raw"]["website"]
        except KeyError as e:
            # If any key is missing, log the error and set the hotel name to "No hotel found"
            print(f"Hotel data missing for {city}, {country}: {e.args[0]}")
            hotel_df.loc[index, "Hotel Name"] = "No hotel found"
            hotel_df.loc[index, "Hotel Address"] = "N/A"
            hotel_df.loc[index, "Hotel Distance"] = "N/A"
            hotel_df.loc[index, "Hotel Website"] = "N/A"
    else:
        # If no features were returned, mark the hotel as "No hotel found"
        hotel_df.loc[index, "Hotel Name"] = "No hotel found"
        hotel_df.loc[index, "Hotel Address"] = "N/A"
        hotel_df.loc[index, "Hotel Distance"] = "N/A"
        hotel_df.loc[index, "Hotel Website"] = "N/A"
    
    # Log the search results for the current city
    print(f"Nearest hotel for {city}, {country}: {hotel_df.loc[index, 'Hotel Name']}")

# Display the updated hotel_df
hotel_df.head()



'Starting hotel search for mashava, ZW'
{'features': [], 'type': 'FeatureCollection'}
Nearest hotel for mashava, ZW: No hotel found
'Starting hotel search for yicheng, CN'
{'features': [{'geometry': {'coordinates': [119.8487386, 31.366866399638486],
                            'type': 'Point'},
               'properties': {'accommodation': {'rooms': 270},
                              'address_line1': 'Le Méridien Yixing',
                              'address_line2': '455 东氿大道, 宜城街道, 214200 '
                                               'Jiangsu, China',
                              'categories': ['accommodation',
                                             'accommodation.hotel'],
                              'city': 'Yixing',
                              'country': 'China',
                              'country_code': 'cn',
                              'datasource': {'attribution': '© OpenStreetMap '
                                                            'contributors'

,City,Country,Lat,Lng,Humidity,Hotel Name,Hotel Address,Hotel Distance,Hotel Website
6,mashava,ZW,-20.0367,30.4823,43,No hotel found,N/A,N/A,N/A
10,yicheng,CN,31.3705,119.8703,40,No hotel found,N/A,N/A,N/A
16,buka,UZ,40.8108,69.1986,52,No hotel found,N/A,N/A,N/A
60,ma'an,CN,26.9775,110.7211,59,No hotel found,N/A,N/A,N/A
67,reggane,DZ,26.7158,0.1714,23,No hotel found,N/A,N/A,N/A


### Step 5: Add the hotel name and the country as additional information in the hover message for each city in the map.

In [19]:
# Merge the hotel information (Hotel Name and Country) into the city_data_df for easier access
city_data_df = city_data_df.merge(hotel_df[['City', 'Hotel Name', 'Country']], on='City', how='left')

# Configure the map plot
map_plot = city_data_df.hvplot.points(
    "Lng",
    "Lat",
    geo=True,
    tiles="OSM",  # Set the map tiles to OpenStreetMap, or you can use another provider like 'CartoDB positron'
    frame_width=800,
    frame_height=600,
    size="Humidity",
    scale=0.5,
    color="City",
    hover_cols=["City", "Country", "Hotel Name", "Humidity"],  # Add hotel name and country to hover message
)

# Display the map plot
map_plot

:Overlay
   .WMTS.I   :WMTS   [Longitude,Latitude]
   .Points.I :Points   [Lng,Lat]   (City,Humidity,Hotel Name)